Trong phiên bản đầu của data cho finetune GNN, subgraph không có cạnh inverse, nên phiên bản này, thêm inverse cho tất cả các cạnh để đảm bảo mọi edge luôn có 2 hướng.

In [1]:
import os
os.chdir("../")

In [2]:
import pickle as pkl
from load_data import DataLoader
import numpy as np
# from utils import convert_relation
from expand_subgraph import ExpandSubgraph
import torch
import random

In [3]:
data = pkl.load(open("data_for_GNN_finetune_another_way.pkl", "rb"))
data[0]['subgraph']

(tensor([    6,    30,    41,  ..., 13704, 13725, 13772]),
 tensor([0, 0, 0,  ..., 0, 0, 0]),
 tensor([[3793,  107, 6381],
         [3793,  155, 3605],
         [3793,  106, 6381],
         ...,
         [3142,  155, 1685],
         [3142,  239, 2016],
         [3142,  155,  803]]))

In [4]:
def add_inverse_edges(subgraph_edges):
    # Add inverse edges for each (start_entity, answer_id) pair
    inverse_edges = []
    for edges in subgraph_edges:
        inv_rel_idx = edges[1] + 1 if edges[1] % 2 == 0 else edges[1] - 1
        inv_edge = torch.tensor([edges[2], inv_rel_idx, edges[0]])
        inverse_edges.append(inv_edge)
    return torch.cat([subgraph_edges, torch.stack(inverse_edges)], dim=0)

In [5]:
new_data = []
for i in range(len(data)):
    subgraph_edges = torch.tensor(data[i]['subgraph'][2])
    subgraph_edges_with_inv = add_inverse_edges(subgraph_edges)
    data[i]['subgraph'] = list(data[i]['subgraph'])
    data[i]['subgraph'][2] = subgraph_edges_with_inv
    data[i]['subgraph'] = tuple(data[i]['subgraph'])
    new_data.append(data[i])

C:\Users\MTBH\AppData\Local\Temp\ipykernel_12012\461315030.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  subgraph_edges = torch.tensor(data[i]['subgraph'][2])


In [11]:
len(new_data[0]['subgraph'][2])

5324

In [ ]:
for i in range(len(new_data[:1])):
    subgraph_edges = torch.tensor(new_data[i]['subgraph'][2])
    for idx, edges in enumerate(subgraph_edges):
        inv_rel_idx = edges[1] + 1 if edges[1] % 2 == 0 else edges[1] - 1
        inv_edge = torch.tensor([edges[2], inv_rel_idx, edges[0]])
        # Check if inv_edge in subgraph_e
        # dges
        # print(idx)
        if not any((subgraph_edges[:,0] == inv_edge[0]) & (subgraph_edges[:,1] == inv_edge[1]) & (subgraph_edges[:,2] == inv_edge[2])):
            # print(f"Missing inverse edge for {edges}: {inv_edge}")
            assert False, f"Missing inverse edge for {edges}: {inv_edge}"

### Output Data

In [12]:
with open("data_for_GNN_finetune_another_way_2.pkl", "wb") as f:
    pkl.dump(new_data, f)

### Test data

In [5]:
new_data = pkl.load(open("data_for_GNN_finetune_another_way_2.pkl", "rb"))
new_data[0]['subgraph']

(tensor([    6,    30,    41,  ..., 13704, 13725, 13772]),
 tensor([0, 0, 0,  ..., 0, 0, 0]),
 tensor([[3793,  107, 6381],
         [3793,  155, 3605],
         [3793,  106, 6381],
         ...,
         [1685,  154, 3142],
         [2016,  238, 3142],
         [ 803,  154, 3142]]))

In [4]:
total_valid_query = 0
total_query = 0
final_data = []
for i, query in enumerate(new_data):
    subgraph_edges = query['subgraph'][2].cpu().numpy()
    subgraph_nodes = query['subgraph'][0].cpu().numpy()
    # print(subgraph_edges.shape)
    drop_edges = query['drop_edges']

    ok = 1
    for e in drop_edges:
        dir_triplet = np.array([e[0], e[1], e[2]])
        assert not np.any(np.all(subgraph_edges == dir_triplet, axis=1)) 
        inv_rel = e[1] + 1 if e[1] % 2 == 0 else e[1] - 1
        inv_triplet = np.array([e[2], inv_rel, e[0]])
        assert not np.any(np.all(subgraph_edges == inv_triplet, axis=1)) 
        # total_valid_query +=  ((e[0] in subgraph_nodes) & (e[2] in subgraph_nodes))
        total_query += 1
        if not ((e[0] in subgraph_nodes) & (e[2] in subgraph_nodes)):
            ok = 0
            # break
    if ok:
        total_valid_query += len(drop_edges)
        final_data.append(query)

print("Total queries: ", total_query)
print("Total valid dropped edges: ", total_valid_query)
print("Valid dropped edges ratio: ", total_valid_query / total_query)
print("Len Subgraph datas: ", len(final_data))

Total queries:  37126
Total valid dropped edges:  37126
Valid dropped edges ratio:  1.0
Len Subgraph datas:  460
